# 05 — Vegetationskantdetektion

[![Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TobiasEdman/space-data-lab/main?labpath=notebooks%2Fsdl3-cases%2F070_CASE_060-Vegetation-Edge.ipynb)

**Syfte:** Kartlägga gränsen mellan vegeterade och icke-vegeterade zoner — användbart för att följa hyggen, skogsåterväxt, dynzontering i kustmiljö och avgränsa våtmarker.

**Partners som bidragit:**
- **Skogsstyrelsen** — hyggeanmälan, återbeskogningsuppföljning
- **Naturvårdsverket** — NMD-referens, våtmarksinventering
- **Stockholms universitet (Inst. för naturgeografi)** — VedgeSat-metodvalidering
- **RISE** — ImintEngine-implementation

**Datakällor:**
- Copernicus Sentinel-2 L2A via Digital Earth Sweden
- NMD 2018 (Naturvårdsverket) — basreferens

**Referensmetod:**
- Muir et al. (2024), *VedgeSat: An automated, open-source toolkit for coastal change monitoring using satellite-derived vegetation edges*, Earth Surf. Process. Landf., 49, 2405–2423
- Nugraha et al. (2026), tropisk validering, Front. Mar. Sci., 13, 1757991

**Licens:** CC0 1.0 (notebook)

## 1. Metod

**NDVI-baserad trösklingspipeline:**

```
Sentinel-2 → NDVI = (NIR − Red) / (NIR + Red)    [B08, B04]
           → Tröskling (Otsu / Weighted Peaks / Fixed)
           → 3-klass: water / non-vegetated / vegetated
           → Morfologisk rensning (opening + closing)
           → Binär vegetationskant-mask (1-pixel centerline)
           → Vektorisering → konturer som GeoJSON
```

**Tre tröskelmetoder** (konfigurerbart i `config/analyzers.yaml`):

| Metod | När |
|-------|-----|
| **Otsu** | Default — automatisk tröskelval per bild |
| **Weighted Peaks** | VedgeSat dual-Gaussian: `Io = 0.2·z_veg + 0.8·z_nonveg` |
| **Fixed** | Manuell tröskel (0.1–0.3 typiskt för svenska kuster) |

## 2. Setup

In [ ]:
from executors.local import LocalExecutor
import matplotlib.pyplot as plt
import folium

# AOI: Småländska kustlandet — mosaik av skog, öppen mark, våtmark
AOI = {
    "west": 16.30,
    "south": 56.95,
    "east": 16.50,
    "north": 57.05,
}
DATE = "2024-07-15"  # Peak vegetation

print(f"AOI: {AOI}")
print(f"Datum: {DATE}")

## 3. Kör analys

In [ ]:
executor = LocalExecutor(
    output_dir="outputs/vegetationskant",
    config_path="config/analyzers.yaml",
)

result = executor.execute(
    date=DATE,
    coords=AOI,
)

if "vegetation_edge" in result.analyses:
    ve = result.analyses["vegetation_edge"].data
    print(f"Tröskel använd: {ve.get('threshold', 'N/A'):.3f}")
    print(f"Metod: {ve.get('method', 'N/A')}")
    stats = ve.get("statistics", {})
    for cls, pct in stats.items():
        print(f"  {cls}: {pct*100:.1f}%")

## 4. Visualisering

In [ ]:
# Interaktiv karta med vegetationskanter som vektorer
center = [(AOI["south"] + AOI["north"]) / 2, (AOI["west"] + AOI["east"]) / 2]
m = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri", name="Satellit",
).add_to(m)

if "vegetation_edge" in result.analyses:
    contours = result.analyses["vegetation_edge"].data.get("contours")
    if contours:
        folium.GeoJson(
            contours,
            style_function=lambda x: {"color": "#00aa00", "weight": 2},
            name="Vegetationskant",
        ).add_to(m)

folium.LayerControl().add_to(m)
m

In [ ]:
# Statiska paneler: RGB, NDVI, segmentering, kant-mask
fig, axes = plt.subplots(2, 2, figsize=(14, 12))

axes[0, 0].imshow(result.rgb)
axes[0, 0].set_title(f"RGB ({DATE})")
axes[0, 0].axis("off")

if "spectral" in result.analyses and "ndvi" in result.analyses["spectral"].data:
    ndvi = result.analyses["spectral"].data["ndvi"]
    im = axes[0, 1].imshow(ndvi, cmap="RdYlGn", vmin=-0.2, vmax=0.9)
    axes[0, 1].set_title("NDVI")
    plt.colorbar(im, ax=axes[0, 1], fraction=0.046)
axes[0, 1].axis("off")

if "vegetation_edge" in result.analyses:
    seg = result.analyses["vegetation_edge"].data.get("segmentation")
    if seg is not None:
        axes[1, 0].imshow(seg, cmap="tab10")
        axes[1, 0].set_title("3-klass: water / non-vegetated / vegetated")
    edges = result.analyses["vegetation_edge"].data.get("edge_mask")
    if edges is not None:
        axes[1, 1].imshow(edges, cmap="binary")
        axes[1, 1].set_title("Vegetationskant-mask (centerline)")
axes[1, 0].axis("off")
axes[1, 1].axis("off")

plt.tight_layout()
plt.show()

## 5. Tolkning

**Användningsområden:**
- **Skogsbruk:** Hyggeskanternas utbredning, återbeskogningsprogress över tid
- **Kustforskning:** Dynvegetationens utbredning och förskjutning (SU-samarbete)
- **Våtmarksförvaltning:** Gräns mellan vassbälten och öppet vatten
- **Jordbruk:** Gräns mellan åker och skog

**Metodjämförelse mot VedgeSat:**
- Samma NDVI-ansats, Weighted Peaks-tröskel direkt från Muir et al. (2024)
- ImintEngine utökar med morfologisk rensning och vektorisering i ett steg

**Nästa steg:**
- Multi-temporal: jämför kanter mellan år → detektera hyggen / återväxt
- Koppla till Skogsstyrelsens avverkningsanmälningar för validering
- Kombinera med NDWI för bättre våtmarksavgränsning
- Prova på tropisk data (jmf. Nugraha et al. 2026)

### Länkar
- [ImintEngine/imint/analyzers/vegetation_edge.py](https://github.com/TobiasEdman/imintengine/blob/main/imint/analyzers/vegetation_edge.py)
- [VedgeSat toolkit (Muir et al., 2024)](https://doi.org/10.1002/esp.5835)
- [Nugraha et al. (2026)](https://doi.org/10.3389/fmars.2026.1757991)